# Orchestrator

Runs notebooks 01–05 in sequence, then merges the results into a single CSV.

```
01_identifiers.ipynb            → csv/01_identifiers.csv
02_y_target.ipynb               → csv/02_y_target.csv
03_morphological.ipynb          → csv/03_morphological.csv
04_synergistic_proximity.ipynb  → csv/04_synergistic_proximity.csv
05_socioeconomic.ipynb          → csv/05_socioeconomic.csv
── [OPTIONAL] ──────────────────────────────────────────
06_joker.ipynb                  → csv/06_joker.csv        (slow — run separately)
```

**Workflow:**
1. Run cells 1–3 → executes notebooks 01–05
2. *(Optional)* Run cell 4 to add median income data
3. Run cell 5 → merges all available CSVs into `csv/final_dataset.csv` and cleans up intermediates

To run: **Restart kernel → Run All Cells** (skips Joker) or run cells individually.

In [ ]:
import papermill as pm
import pandas as pd
import pathlib, time, os, tempfile
from datetime import datetime

os.makedirs("csv", exist_ok=True)

RUN_ID = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
print(f"papermill {pm.__version__}")
print(f"Run ID: {RUN_ID}")

In [ ]:
# ── Pipeline (notebooks 01–05) ────────────────────────
PIPELINE = [
    ("01_identifiers.ipynb",           "01 · Identifiers"),
    ("02_y_target.ipynb",              "02 · Y-Target"),
    ("03_morphological.ipynb",         "03 · Morphological"),
    ("04_synergistic_proximity.ipynb", "04 · Synergistic Proximity"),
    ("05_socioeconomic.ipynb",         "05 · Socioeconomic"),
]

# Set any to False to skip manually
RUN_MODULE = {nb: True for nb, _ in PIPELINE}

KERNEL_NAME = "python3"  # change to "osmnx-scraper" if using the custom kernel

print("Pipeline (01–05):")
for nb, desc in PIPELINE:
    status = "RUN" if RUN_MODULE[nb] else "SKIP"
    print(f"  [{status}]  {desc}")
print("\n  [SKIP]  06 · Joker  ← run cell 4 separately if needed")

In [ ]:
# ── Execute notebooks ─────────────────────────────────
results = []

for nb_path, description in PIPELINE:
    if not RUN_MODULE.get(nb_path, True):
        print(f"  SKIP  {description}")
        results.append((nb_path, "skipped", 0))
        continue

    if not pathlib.Path(nb_path).exists():
        raise FileNotFoundError(f"Notebook not found: {nb_path}")

    print(f"\n{'='*55}")
    print(f"  {description}")
    print(f"{'='*55}")

    tmp_path = pathlib.Path(tempfile.mktemp(suffix=".ipynb"))
    t0 = time.time()
    try:
        pm.execute_notebook(
            nb_path,
            str(tmp_path),
            kernel_name=KERNEL_NAME,
            progress_bar=False,
        )
        elapsed = time.time() - t0
        print(f"  Status : OK  ({elapsed:.1f} s)")
        results.append((nb_path, "ok", elapsed))

    except pm.exceptions.PapermillExecutionError as e:
        elapsed = time.time() - t0
        print(f"  Status : FAILED  ({elapsed:.1f} s)")
        print(f"  Error  : {e}")
        results.append((nb_path, "failed", elapsed))
        print("\nPipeline stopped. Fix the error and re-run.")
        break

    finally:
        if tmp_path.exists():
            tmp_path.unlink()

# ── Summary ───────────────────────────────────────────
print(f"\n{'='*55}")
print("  PIPELINE SUMMARY")
print(f"{'='*55}")
for nb_path, status, elapsed in results:
    icon = {"ok": "OK", "failed": "FAIL", "skipped": "SKIP"}.get(status, "?")
    t_str = f"{elapsed:.1f} s" if elapsed else "-"
    print(f"  [{icon:4s}]  {nb_path:<42s} {t_str}")
print(f"{'='*55}")

## [Optional] Cell 4 — Joker: Median Income
Run this cell **only if you want median income data**. It can take several minutes due to Census API rate limits.
Skip it and jump directly to Cell 5 (merge) if you don't need it.

In [ ]:
# ── [OPTIONAL] Joker — run this cell manually if needed ──
# Executes 06_joker.ipynb and produces csv/06_joker.csv
# Skip this cell and go directly to the merge if you don't need median income.

print("Running 06 · Joker (median income)...")
print("This may take several minutes due to Census API rate limits.\n")

tmp_path = pathlib.Path(tempfile.mktemp(suffix=".ipynb"))
t0 = time.time()
try:
    pm.execute_notebook(
        "06_joker.ipynb",
        str(tmp_path),
        kernel_name=KERNEL_NAME,
        progress_bar=False,
    )
    elapsed = time.time() - t0
    print(f"  Status : OK  ({elapsed:.1f} s)")
    print(f"  Output : csv/06_joker.csv")
except pm.exceptions.PapermillExecutionError as e:
    print(f"  Status : FAILED  — {e}")
    print("  csv/06_joker.csv will be skipped in the merge step.")
finally:
    if tmp_path.exists():
        tmp_path.unlink()

In [ ]:
# ── Cell 5 · Merge + cleanup ──────────────────────────
print("Merging outputs...\n")

CSV_GROUPS = [
    ("csv/01_identifiers.csv",           ["osm_id", "lat", "lon"]),
    ("csv/02_y_target.csv",              ["amenity_label"]),
    ("csv/03_morphological.csv",         ["street_width_m", "plot_area_m2"]),
    ("csv/04_synergistic_proximity.csv", ["dist_bus_stop_m", "dist_hospital_m", "dist_school_m", "dist_park_m"]),
    ("csv/05_socioeconomic.csv",         ["residential_ratio", "commercial_ratio"]),
    ("csv/06_joker.csv",                 ["median_income"]),
]

if not pathlib.Path("csv/01_identifiers.csv").exists():
    raise FileNotFoundError("csv/01_identifiers.csv not found — run notebooks 01–05 first.")

df_final = pd.read_csv("csv/01_identifiers.csv")

for csv_path, cols in CSV_GROUPS[1:]:
    if pathlib.Path(csv_path).exists():
        df_other = pd.read_csv(csv_path)
        df_final = df_final.merge(df_other, on="osm_id", how="left")
        print(f"  OK    {csv_path}")
    else:
        for col in cols:
            df_final[col] = None
        print(f"  WARN  {csv_path} not found — columns filled with NaN")

FINAL_COLUMNS = [
    "osm_id", "lat", "lon",
    "amenity_label",
    "street_width_m", "plot_area_m2",
    "dist_bus_stop_m", "dist_hospital_m", "dist_school_m", "dist_park_m",
    "residential_ratio", "commercial_ratio",
    "median_income",
]
df_final = df_final[[c for c in FINAL_COLUMNS if c in df_final.columns]]

# Find the next version number based on existing final_dataset files
existing = sorted(pathlib.Path("csv").glob("final_dataset_v*.csv"))
next_version = len(existing) + 1
output_name = f"final_dataset_v{next_version:03d}_{RUN_ID}.csv"
output_path = f"csv/{output_name}"

df_final.to_csv(output_path, index=False, encoding="utf-8")

print(f"\nFinal dataset: {df_final.shape[0]} rows x {df_final.shape[1]} columns")
print(f"Saved: {output_path}")

# ── Cleanup intermediate CSVs ─────────────────────────
INTERMEDIATES = [
    "csv/00_base_data.csv",
    "csv/01_identifiers.csv",
    "csv/02_y_target.csv",
    "csv/03_morphological.csv",
    "csv/04_synergistic_proximity.csv",
    "csv/05_socioeconomic.csv",
    "csv/06_joker.csv",
]
print("\nCleaning up intermediate files...")
for f in INTERMEDIATES:
    p = pathlib.Path(f)
    if p.exists():
        p.unlink()
        print(f"  Deleted {f}")
print("Done — only", output_name, "remains.")

In [ ]:
# ── Preview ───────────────────────────────────────────
print(df_final.dtypes)
print()
df_final.head(10)

In [ ]:
# ── Column completeness ──────────────────────────────
print("Column completeness:")
for col in FINAL_COLUMNS:
    n = df_final[col].notna().sum()
    pct = 100 * n / len(df_final)
    print(f"  {col:<22s} {n:>5d}/{len(df_final)}  ({pct:.1f}%)")